In [0]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn as sklearn
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE

In [0]:
%pip install imbalanced-learn

In [0]:
# Carrega os dados da tabela Unity Catalog
df = spark.table('workspace.default.creditcard').toPandas()

In [0]:
df.info()

In [0]:
df.head()

In [0]:
df.isna().sum()

In [0]:
df.dropna(inplace=True)

In [0]:
df.corr()['Class'].sort_values(ascending=False)

In [0]:
df.describe()

In [0]:
#Separando dados para treinamento

x=df.drop(['Class'],axis=1)
y=df['Class']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.3,random_state=7)

ESCOLHER O MODELO DE BALANCEAMENTO MAIS ADEQUADO AO PROBLEMA

In [0]:
# Aplicar SMOTE para balancear as classes (apenas no treino)
print("Distribuição ANTES do SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=7)
x_train_balanced, y_train_balanced = smote.fit_resample(x_train, y_train)

print("\nDistribuição DEPOIS do SMOTE:")
print(pd.Series(y_train_balanced).value_counts())
print("\nProporção balanceada:")
print(pd.Series(y_train_balanced).value_counts(normalize=True))

In [0]:
# ADASYN - Gera mais exemplos sintéticos em regiões difíceis de classificar
# Melhor quando há sobreposição complexa entre classes
from imblearn.over_sampling import ADASYN

print("Distribuição ANTES do ADASYN:")
print(y_train.value_counts())

adasyn = ADASYN(random_state=7)
x_train_balanced, y_train_balanced = adasyn.fit_resample(x_train, y_train)

print("\nDistribuição DEPOIS do ADASYN:")
print(pd.Series(y_train_balanced).value_counts())
print("\nProporção balanceada:")
print(pd.Series(y_train_balanced).value_counts(normalize=True))

In [0]:
# BorderlineSMOTE - Gera exemplos sintéticos apenas na fronteira entre classes
# Útil quando a separação entre classes é sutil
from imblearn.over_sampling import BorderlineSMOTE

print("Distribuição ANTES do BorderlineSMOTE:")
print(y_train.value_counts())

borderline_smote = BorderlineSMOTE(random_state=7)
x_train_balanced, y_train_balanced = borderline_smote.fit_resample(x_train, y_train)

print("\nDistribuição DEPOIS do BorderlineSMOTE:")
print(pd.Series(y_train_balanced).value_counts())
print("\nProporção balanceada:")
print(pd.Series(y_train_balanced).value_counts(normalize=True))

In [0]:
# SVMSMOTE - Usa SVM para identificar exemplos de fronteira antes de gerar sintéticos
# Mais sofisticado, bom para dados com ruído
from imblearn.over_sampling import SVMSMOTE

print("Distribuição ANTES do SVMSMOTE:")
print(y_train.value_counts())

svm_smote = SVMSMOTE(random_state=7)
x_train_balanced, y_train_balanced = svm_smote.fit_resample(x_train, y_train)

print("\nDistribuição DEPOIS do SVMSMOTE:")
print(pd.Series(y_train_balanced).value_counts())
print("\nProporção balanceada:")
print(pd.Series(y_train_balanced).value_counts(normalize=True))

In [0]:
# SMOTETomek - HÍBRIDO: Gera sintéticos com SMOTE E limpa a fronteira com Tomek Links
# RECOMENDADO para detecção de fraude!
from imblearn.combine import SMOTETomek

print("Distribuição ANTES do SMOTETomek:")
print(y_train.value_counts())

smote_tomek = SMOTETomek(random_state=7)
x_train_balanced, y_train_balanced = smote_tomek.fit_resample(x_train, y_train)

print("\nDistribuição DEPOIS do SMOTETomek:")
print(pd.Series(y_train_balanced).value_counts())
print("\nProporção balanceada:")
print(pd.Series(y_train_balanced).value_counts(normalize=True))

In [0]:
# SMOTEENN - HÍBRIDO: SMOTE + Edited Nearest Neighbours
# Limpeza mais agressiva que SMOTETomek, bom para dados muito ruidosos
from imblearn.combine import SMOTEENN

print("Distribuição ANTES do SMOTEENN:")
print(y_train.value_counts())

smote_enn = SMOTEENN(random_state=7)
x_train_balanced, y_train_balanced = smote_enn.fit_resample(x_train, y_train)

print("\nDistribuição DEPOIS do SMOTEENN:")
print(pd.Series(y_train_balanced).value_counts())
print("\nProporção balanceada:")
print(pd.Series(y_train_balanced).value_counts(normalize=True))

In [0]:
# RandomOverSampler - Duplica aleatoriamente exemplos da classe minoritária
# Simples e rápido, mas pode causar overfitting
from imblearn.over_sampling import RandomOverSampler

print("Distribuição ANTES do RandomOverSampler:")
print(y_train.value_counts())

ros = RandomOverSampler(random_state=7)
x_train_balanced, y_train_balanced = ros.fit_resample(x_train, y_train)

print("\nDistribuição DEPOIS do RandomOverSampler:")
print(pd.Series(y_train_balanced).value_counts())
print("\nProporção balanceada:")
print(pd.Series(y_train_balanced).value_counts(normalize=True))

ESCOLHA O MÉTODO DE COMPARAÇÃO DOS MODELOS DE BALANCEAMENTO

In [0]:
# STRATIFIED K-FOLD CV - Padrão com 5 folds
# Mantém proporção de fraudes em cada fold
# Bom equilíbrio entre robustez e velocidade

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Dicionário com os 3 balanceadores selecionados
balanceadores = {
    'SMOTE': SMOTE(random_state=7),
    'SMOTETomek': SMOTETomek(random_state=7),
    'SMOTEENN': SMOTEENN(random_state=7)
}

# Configurar Cross-Validation com 5 folds (mantém proporção de fraudes)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)

# Definir métricas de avaliação
scoring = {
    'precision': make_scorer(precision_score, pos_label=1, zero_division=0),
    'recall': make_scorer(recall_score, pos_label=1, zero_division=0),
    'f1': make_scorer(f1_score, pos_label=1, zero_division=0),
    'roc_auc': 'roc_auc'
}

# Armazenar resultados
resultados = []

print("🔄 Testando com 5-Fold Stratified Cross-Validation...\n")
print("="*80)

for nome, balanceador in balanceadores.items():
    print(f"\n>>> Testando {nome}...")
    
    # Criar pipeline: balanceador + modelo
    pipeline = Pipeline([
        ('balanceador', balanceador),
        ('modelo', RandomForestClassifier(random_state=7))
    ])
    
    # Executar Cross-Validation
    cv_results = cross_validate(
        pipeline, x_train, y_train, 
        cv=skf, 
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1  # Usa todos os cores disponíveis
    )
    
    # Calcular média e desvio padrão das métricas
    precision_mean = cv_results['test_precision'].mean()
    precision_std = cv_results['test_precision'].std()
    
    recall_mean = cv_results['test_recall'].mean()
    recall_std = cv_results['test_recall'].std()
    
    f1_mean = cv_results['test_f1'].mean()
    f1_std = cv_results['test_f1'].std()
    
    auc_mean = cv_results['test_roc_auc'].mean()
    auc_std = cv_results['test_roc_auc'].std()
    
    # Armazenar resultados
    resultados.append({
        'Método': nome,
        'Precision': f"{precision_mean:.3f} ± {precision_std:.3f}",
        'Recall': f"{recall_mean:.3f} ± {recall_std:.3f}",
        'F1-Score': f"{f1_mean:.3f} ± {f1_std:.3f}",
        'AUC': f"{auc_mean:.3f} ± {auc_std:.3f}",
        'F1_numeric': f1_mean  # Para ordenação
    })
    
    print(f"   ✓ Precision: {precision_mean:.3f} (±{precision_std:.3f})")
    print(f"   ✓ Recall: {recall_mean:.3f} (±{recall_std:.3f})")
    print(f"   ✓ F1-Score: {f1_mean:.3f} (±{f1_std:.3f})")
    print(f"   ✓ AUC: {auc_mean:.3f} (±{auc_std:.3f})")

print("\n" + "="*80)
print("\n🏆 RANKING DOS BALANCEADORES (ordenado por F1-Score médio):\n")

# Criar DataFrame de resultados
df_resultados = pd.DataFrame(resultados).sort_values('F1_numeric', ascending=False)
df_resultados_display = df_resultados.drop('F1_numeric', axis=1)
print(df_resultados_display.to_string(index=False))

print(f"\n💡 Melhor método: {df_resultados.iloc[0]['Método']}")
print("\n📊 Nota: Valores mostrados como 'média ± desvio padrão' dos 5 folds")

In [0]:
# TIME SERIES SPLIT - Valida respeitando ordem temporal
# Treina em dados PASSADOS, valida em dados FUTUROS (simula produção real)
# IMPORTANTE: Seus dados têm coluna 'Time' - isso é crucial!

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Dicionário com os 3 balanceadores selecionados
balanceadores = {
    'SMOTE': SMOTE(random_state=7),
    'SMOTETomek': SMOTETomek(random_state=7),
    'SMOTEENN': SMOTEENN(random_state=7)
}

# Configurar Time Series Split com 5 splits
# Cada split usa dados cronologicamente anteriores para treino
tscv = TimeSeriesSplit(n_splits=5)

# Armazenar resultados
resultados = []

print("⏰ Testando com Time Series Split (ordem temporal)...\n")
print("="*80)
print("⚡ Simula cenário real: treina no passado, prevê o futuro!\n")

for nome, balanceador in balanceadores.items():
    print(f"\n>>> Testando {nome}...")
    
    # Listas para armazenar métricas de cada split
    precision_scores = []
    recall_scores = []
    f1_scores = []
    auc_scores = []
    
    # Iterar sobre cada split temporal
    for fold_idx, (train_idx, val_idx) in enumerate(tscv.split(x_train), 1):
        # Dividir dados respeitando ordem temporal
        x_train_fold = x_train.iloc[train_idx]
        y_train_fold = y_train.iloc[train_idx]
        x_val_fold = x_train.iloc[val_idx]
        y_val_fold = y_train.iloc[val_idx]
        
        # Aplicar balanceamento no fold de treino
        x_train_balanced_fold, y_train_balanced_fold = balanceador.fit_resample(x_train_fold, y_train_fold)
        
        # Treinar modelo
        modelo_temp = RandomForestClassifier(random_state=7)
        modelo_temp.fit(x_train_balanced_fold, y_train_balanced_fold)
        
        # Fazer previsões no fold de validação
        y_pred_fold = modelo_temp.predict(x_val_fold)
        
        # Calcular métricas
        precision_scores.append(precision_score(y_val_fold, y_pred_fold, pos_label=1, zero_division=0))
        recall_scores.append(recall_score(y_val_fold, y_pred_fold, pos_label=1, zero_division=0))
        f1_scores.append(f1_score(y_val_fold, y_pred_fold, pos_label=1, zero_division=0))
        auc_scores.append(roc_auc_score(y_val_fold, y_pred_fold))
    
    # Calcular média e desvio padrão
    precision_mean = np.mean(precision_scores)
    precision_std = np.std(precision_scores)
    
    recall_mean = np.mean(recall_scores)
    recall_std = np.std(recall_scores)
    
    f1_mean = np.mean(f1_scores)
    f1_std = np.std(f1_scores)
    
    auc_mean = np.mean(auc_scores)
    auc_std = np.std(auc_scores)
    
    # Armazenar resultados
    resultados.append({
        'Método': nome,
        'Precision': f"{precision_mean:.3f} ± {precision_std:.3f}",
        'Recall': f"{recall_mean:.3f} ± {recall_std:.3f}",
        'F1-Score': f"{f1_mean:.3f} ± {f1_std:.3f}",
        'AUC': f"{auc_mean:.3f} ± {auc_std:.3f}",
        'F1_numeric': f1_mean
    })
    
    print(f"   ✓ Precision: {precision_mean:.3f} (±{precision_std:.3f})")
    print(f"   ✓ Recall: {recall_mean:.3f} (±{recall_std:.3f})")
    print(f"   ✓ F1-Score: {f1_mean:.3f} (±{f1_std:.3f})")
    print(f"   ✓ AUC: {auc_mean:.3f} (±{auc_std:.3f})")

print("\n" + "="*80)
print("\n🏆 RANKING DOS BALANCEADORES (ordenado por F1-Score médio):\n")

df_resultados = pd.DataFrame(resultados).sort_values('F1_numeric', ascending=False)
df_resultados_display = df_resultados.drop('F1_numeric', axis=1)
print(df_resultados_display.to_string(index=False))

print(f"\n💡 Melhor método: {df_resultados.iloc[0]['Método']}")
print("\n📊 Nota: Validação temporal - treina em dados ANTIGOS, valida em dados RECENTES")

In [0]:
# REPEATED STRATIFIED K-FOLD - Repete CV múltiplas vezes
# MAIS ROBUSTO que CV simples: reduz variância dos resultados
# 5-Fold repetido 3 vezes = 15 avaliações totais!

from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

# Dicionário com os 3 balanceadores selecionados
balanceadores = {
    'SMOTE': SMOTE(random_state=7),
    'SMOTETomek': SMOTETomek(random_state=7),
    'SMOTEENN': SMOTEENN(random_state=7)
}

# Configurar Repeated Stratified K-Fold: 5 folds repetidos 3 vezes
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=7)

# Definir métricas de avaliação
scoring = {
    'precision': make_scorer(precision_score, pos_label=1, zero_division=0),
    'recall': make_scorer(recall_score, pos_label=1, zero_division=0),
    'f1': make_scorer(f1_score, pos_label=1, zero_division=0),
    'roc_auc': 'roc_auc'
}

# Armazenar resultados
resultados = []

print("🔄 Testando com Repeated Stratified K-Fold (5-Fold × 3 repetições)...\n")
print("="*80)
print("⚡ MAIS ROBUSTO: cada método é avaliado 15 vezes!\n")

for nome, balanceador in balanceadores.items():
    print(f"\n>>> Testando {nome}...")
    
    # Criar pipeline: balanceador + modelo
    pipeline = Pipeline([
        ('balanceador', balanceador),
        ('modelo', RandomForestClassifier(random_state=7))
    ])
    
    # Executar Repeated Cross-Validation
    cv_results = cross_validate(
        pipeline, x_train, y_train,
        cv=rskf,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    
    # Calcular média e desvio padrão das métricas
    precision_mean = cv_results['test_precision'].mean()
    precision_std = cv_results['test_precision'].std()
    
    recall_mean = cv_results['test_recall'].mean()
    recall_std = cv_results['test_recall'].std()
    
    f1_mean = cv_results['test_f1'].mean()
    f1_std = cv_results['test_f1'].std()
    
    auc_mean = cv_results['test_roc_auc'].mean()
    auc_std = cv_results['test_roc_auc'].std()
    
    # Armazenar resultados
    resultados.append({
        'Método': nome,
        'Precision': f"{precision_mean:.3f} ± {precision_std:.3f}",
        'Recall': f"{recall_mean:.3f} ± {recall_std:.3f}",
        'F1-Score': f"{f1_mean:.3f} ± {f1_std:.3f}",
        'AUC': f"{auc_mean:.3f} ± {auc_std:.3f}",
        'F1_numeric': f1_mean
    })
    
    print(f"   ✓ Precision: {precision_mean:.3f} (±{precision_std:.3f})")
    print(f"   ✓ Recall: {recall_mean:.3f} (±{recall_std:.3f})")
    print(f"   ✓ F1-Score: {f1_mean:.3f} (±{f1_std:.3f})")
    print(f"   ✓ AUC: {auc_mean:.3f} (±{auc_std:.3f})")

print("\n" + "="*80)
print("\n🏆 RANKING DOS BALANCEADORES (ordenado por F1-Score médio):\n")

df_resultados = pd.DataFrame(resultados).sort_values('F1_numeric', ascending=False)
df_resultados_display = df_resultados.drop('F1_numeric', axis=1)
print(df_resultados_display.to_string(index=False))

print(f"\n💡 Melhor método: {df_resultados.iloc[0]['Método']}")
print("\n📊 Nota: 15 avaliações totais (5 folds × 3 repetições) - resultados ultra-robustos!")

ESCOLHA O MODELO PREDITIVO QUE MAIS SE ADEQUA AO PROBLEMA

In [0]:
%pip install xgboost lightgbm

In [0]:
# RANDOM FOREST - Padrão, robusto, menos overfitting
# Bom equilíbrio entre performance e simplicidade

from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,      # Número de árvores
    max_depth=20,          # Profundidade máxima
    min_samples_split=10,  # Mínimo de amostras para dividir
    min_samples_leaf=4,    # Mínimo de amostras nas folhas
    random_state=7,
    n_jobs=-1,             # Usa todos os cores
    class_weight='balanced' # Ajusta pesos por classe
)

print("✓ Modelo Random Forest criado!")
print(f"Parâmetros: {model.get_params()}")

In [0]:
# XGBOOST - MELHOR para detecção de fraude!
# Estado da arte para dados tabulares

import xgboost as xgb
from sklearn.base import BaseEstimator, ClassifierMixin

# Calcular peso para classe desbalanceada
scale_pos_weight = (y_train_balanced == 0).sum() / (y_train_balanced == 1).sum()

model = xgb.XGBClassifier(
    n_estimators=100,           # Número de árvores
    max_depth=6,                # Profundidade máxima
    learning_rate=0.1,          # Taxa de aprendizado
    subsample=0.8,              # Fração de amostras por árvore
    colsample_bytree=0.8,       # Fração de features por árvore
    scale_pos_weight=scale_pos_weight,  # Peso para classe minoritária
    random_state=7,
    n_jobs=-1,
    eval_metric='auc'
)

print("✓ Modelo XGBoost criado!")
print(f"Scale pos weight: {scale_pos_weight:.2f}")
print(f"Parâmetros principais: n_estimators={model.n_estimators}, max_depth={model.max_depth}, lr={model.learning_rate}")

In [0]:
# LIGHTGBM - MUITO rápido, eficiente
# Comparável ao XGBoost, consome menos memória

import lightgbm as lgb

# Calcular peso para classe desbalanceada
scale_pos_weight = (y_train_balanced == 0).sum() / (y_train_balanced == 1).sum()

model = lgb.LGBMClassifier(
    n_estimators=100,           # Número de árvores
    max_depth=6,                # Profundidade máxima
    learning_rate=0.1,          # Taxa de aprendizado
    subsample=0.8,              # Fração de amostras
    colsample_bytree=0.8,       # Fração de features
    scale_pos_weight=scale_pos_weight,  # Peso para classe minoritária
    random_state=7,
    n_jobs=-1,
    verbose=-1                  # Silencioso
)

print("✓ Modelo LightGBM criado!")
print(f"Scale pos weight: {scale_pos_weight:.2f}")
print(f"Parâmetros principais: n_estimators={model.n_estimators}, max_depth={model.max_depth}, lr={model.learning_rate}")

In [0]:
# GRADIENT BOOSTING - Clássico sklearn, sem libs extras
# Muito bom, mais lento que XGBoost/LightGBM

from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=100,           # Número de árvores
    max_depth=6,                # Profundidade máxima
    learning_rate=0.1,          # Taxa de aprendizado
    subsample=0.8,              # Fração de amostras
    min_samples_split=10,       # Mínimo para dividir
    min_samples_leaf=4,         # Mínimo nas folhas
    random_state=7,
    verbose=0
)

print("✓ Modelo Gradient Boosting criado!")
print(f"Parâmetros principais: n_estimators={model.n_estimators}, max_depth={model.max_depth}, lr={model.learning_rate}")

In [0]:
# LOGISTIC REGRESSION - Baseline simples, rápido, interpretável
# Útil para comparação com modelos mais complexos

from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,              # Máximo de iterações
    class_weight='balanced',    # Ajusta pesos por classe
    random_state=7,
    n_jobs=-1,
    solver='lbfgs'              # Solver para otimização
)

print("✓ Modelo Logistic Regression criado!")
print(f"Parâmetros: max_iter={model.max_iter}, class_weight={model.class_weight}")

In [0]:
# COMPARAR TODOS OS MODELOS PREDITIVOS
# Usa o x_train_balanced e y_train_balanced do balanceador que você escolheu
# Treina e avalia cada modelo no conjunto de teste

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, precision_score, recall_score, f1_score
import xgboost as xgb
import lightgbm as lgb
import pandas as pd
import time

# Calcular peso para classe desbalanceada
scale_pos_weight = (y_train_balanced == 0).sum() / (y_train_balanced == 1).sum()

# Dicionário com todos os modelos
modelos = {
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=20, min_samples_split=10,
        min_samples_leaf=4, random_state=7, n_jobs=-1, class_weight='balanced'
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight,
        random_state=7, n_jobs=-1, eval_metric='auc'
    ),
    'LightGBM': lgb.LGBMClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=scale_pos_weight,
        random_state=7, n_jobs=-1, verbose=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        subsample=0.8, random_state=7, verbose=0
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000, class_weight='balanced',
        random_state=7, n_jobs=-1, solver='lbfgs'
    )
}

# Armazenar resultados
resultados = []

print("🤖 Comparando TODOS os modelos preditivos...\n")
print("="*80)

for nome, modelo in modelos.items():
    print(f"\n>>> Testando {nome}...")
    
    # Medir tempo de treinamento
    start_time = time.time()
    
    # Treinar modelo
    modelo.fit(x_train_balanced, y_train_balanced)
    
    # Tempo de treinamento
    train_time = time.time() - start_time
    
    # Fazer previsões
    y_pred = modelo.predict(x_test)
    
    # Calcular métricas
    precision = precision_score(y_test, y_pred, pos_label=1, zero_division=0)
    recall = recall_score(y_test, y_pred, pos_label=1, zero_division=0)
    f1 = f1_score(y_test, y_pred, pos_label=1, zero_division=0)
    auc = roc_auc_score(y_test, y_pred)
    
    # Armazenar resultados
    resultados.append({
        'Modelo': nome,
        'Precision': f"{precision:.3f}",
        'Recall': f"{recall:.3f}",
        'F1-Score': f"{f1:.3f}",
        'AUC': f"{auc:.3f}",
        'Tempo (s)': f"{train_time:.2f}",
        'F1_numeric': f1  # Para ordenação
    })
    
    print(f"   ✓ Precision: {precision:.3f} | Recall: {recall:.3f} | F1: {f1:.3f} | AUC: {auc:.3f}")
    print(f"   ⏱️ Tempo de treino: {train_time:.2f}s")

print("\n" + "="*80)
print("\n🏆 RANKING DOS MODELOS (ordenado por F1-Score para Fraude):\n")

# Criar DataFrame de resultados
df_resultados = pd.DataFrame(resultados).sort_values('F1_numeric', ascending=False)
df_resultados_display = df_resultados.drop('F1_numeric', axis=1)
print(df_resultados_display.to_string(index=False))

print(f"\n💡 Melhor modelo: {df_resultados.iloc[0]['Modelo']}")
print(f"\n🚀 Mais rápido: {df_resultados.loc[df_resultados['Tempo (s)'].astype(float).idxmin(), 'Modelo']}")

In [0]:
#Treinando Modelo
model = model.fit(x_train_balanced, y_train_balanced)

In [0]:
#Passando dados de teste para o modelo
y_predict = model.predict(x_test)

In [0]:
# Comparando gabarito e Previsoes_da_Maquina
gabarito = pd.DataFrame({'Gabarito': y_test, 'Previsoes_da_Maquina': y_predict})
gabarito

In [0]:
# Evaluate model
print('Classification metrics: \n', classification_report(y_test, y_predict))

In [0]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, y_predict)
print('AUC:', auc)

In [0]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Calcular matriz de confusão
cm = confusion_matrix(y_test, y_predict)

# Plotar matriz de confusão
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Normal', 'Fraude'],
            yticklabels=['Normal', 'Fraude'])
plt.title('Matriz de Confusão', fontsize=16)
plt.ylabel('Valor Real', fontsize=12)
plt.xlabel('Valor Previsto', fontsize=12)
plt.show()

print(f"\nVerdadeiros Negativos (TN): {cm[0,0]}")
print(f"Falsos Positivos (FP): {cm[0,1]}")
print(f"Falsos Negativos (FN): {cm[1,0]}")
print(f"Verdadeiros Positivos (TP): {cm[1,1]}")

In [0]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

# Obter probabilidades para a classe positiva (fraude)
y_proba = model.predict_proba(x_test)[:, 1]

# Calcular curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

# Plotar curva ROC
plt.figure(figsize=(10, 7))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Acaso (AUC = 0.5)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
plt.title('Curva ROC - Receiver Operating Characteristic', fontsize=16)
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.show()

print(f"\nAUC (Area Under the Curve): {roc_auc:.4f}")
print("Quanto mais próximo de 1.0, melhor o modelo em distinguir as classes!")

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# Obter importância das features
feature_importance = pd.DataFrame({
    'feature': x_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Top 15 features mais importantes
top_features = feature_importance.head(15)

# Plotar
plt.figure(figsize=(12, 8))
plt.barh(range(len(top_features)), top_features['importance'], color='steelblue')
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importância', fontsize=12)
plt.ylabel('Features', fontsize=12)
plt.title('Top 15 Features Mais Importantes para Detecção de Fraude', fontsize=16)
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.show()

print("\nTop 10 Features Mais Importantes:")
print(feature_importance.head(10).to_string(index=False))